# Figures

Figures for saved runs (`scripts/train.py`) and, for comparison, for thesis-era result pickles.
To export every figure of a run without a notebook: `uv run python scripts/make_figures.py <run dir>`.

In [ ]:
import logging
from pathlib import Path

import numpy as np

from pinn_jc.evaluation import evaluate, evaluate_run
from pinn_jc.io.legacy import load_legacy_pickle
from pinn_jc.io.runs import load_run
from pinn_jc.training.config import PhysicsConfig
from pinn_jc.visualization import plots

logging.basicConfig(level=logging.INFO)
plots.use_style()  # SciencePlots + LaTeX; use_style(usetex=False) without LaTeX
RUNS = Path("../saved_artifacts/runs")

## A saved run

Pick an experiment output directory and a seed. Predictions are compared with the exact solution in the frame of the picture the run was trained in.

In [ ]:
run_dir = sorted((RUNS / "atom_fock").glob("*"))[-1] / "seed_42"  # latest run of atom_fock
run = load_run(run_dir)
evaluation = evaluate_run(run)
print(run.config.physics.picture, "| final metrics:", run.metrics)

In [ ]:
plots.plot_expectations(evaluation);

In [ ]:
plots.plot_fidelity(evaluation);

In [ ]:
plots.plot_state_components(evaluation);

In [ ]:
plots.plot_losses(run.history, every=10);

In [ ]:
if run.couplings is not None:
    plots.plot_couplings(run.history, run.config.physics.picture, run.config.physics.params, every=10)

## A thesis-era result pickle

These files do not record their configuration: state the physics they were trained with.

In [ ]:
legacy = load_legacy_pickle("../saved_artifacts/training/results_dict_fock_single_shot.pkl")
legacy_physics = PhysicsConfig(
    picture="atom", params={"wc": 1.0, "wa": 1.0, "g1": 1.0}, init_state="fock", dims={"atom": 2, "field": 2}
)
legacy_evaluation = evaluate(legacy["models"], legacy_physics, tfinal=2 * np.pi, n_time_steps=200)
plots.plot_expectations(legacy_evaluation)
plots.plot_couplings(legacy["loss"], "atom", legacy_physics.params, every=10);

## Positivity transforms for the learned couplings

In [ ]:
plots.plot_positivity_transforms();